# Mô phỏng: Noise của mini-batch SGD xấp xỉ Gaussian
Notebook này mô phỏng noise của mini-batch SGD để xem noise có gần Gaussian không.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

def gen(n=20000, d=10):
    X = np.random.randn(n, d)
    w0 = np.random.randn(d)
    y = X @ w0 + np.random.randn(n)
    return X, y, w0

def g_full(X, y, w):
    return (X.T @ (X @ w - y)) / X.shape[0]

def g_mb(X, y, w, b=64):
    idx = np.random.randint(0, X.shape[0], b)
    Xb, yb = X[idx], y[idx]
    return (Xb.T @ (Xb @ w - yb)) / b

X, y, wtrue = gen()
w = np.random.randn(X.shape[1])
gf = g_full(X, y, w)
gf

In [ ]:
noise = []
for _ in range(3000):
    noise.append(g_mb(X, y, w, 64)[0] - gf[0])
noise = np.array(noise)
noise.mean(), noise.std()

In [ ]:
mu, sd = noise.mean(), noise.std()
fig, ax = plt.subplots(figsize=(7,4))
ax.hist(noise, bins=40, density=True, alpha=0.6)

xs = np.linspace(noise.min(), noise.max(), 200)
pdf = 1/(sd*np.sqrt(2*np.pi)) * np.exp(-0.5*((xs-mu)/sd)**2)
ax.plot(xs, pdf)
plt.title("Histogram noise vs Gaussian PDF")
plt.show()

In [ ]:
def qq(samples):
    s = np.sort(samples)
    n = len(s)
    probs = (np.arange(1, n+1) - 0.5) / n
    theo = np.sqrt(2) * np.erfinv(2*probs - 1)
    s_norm = (s - s.mean()) / s.std()
    plt.figure(figsize=(6,6))
    plt.scatter(theo, s_norm, s=5)
    m = min(theo.min(), s_norm.min())
    M = max(theo.max(), s_norm.max())
    plt.plot([m, M], [m, M])
    plt.title("Q-Q Plot noise vs Normal")
    plt.xlabel("Theo quantile")
    plt.ylabel("Sample quantile")
    plt.show()

qq(noise)